# BERT и NER — пример

<a id="libraries"></a>

## Импорт библиотек, выполнение базовых настроек

In [3]:
! rm -f requirements.txt
! python --version

Python 3.12.9


In [ ]:
from pathlib import Path

requirements_file = Path('requirements.txt')
requirements = [
    'torch==2.9.1',
    'transformers==4.46.3',
    'tokenizers==0.20.3',
    'sentencepiece==0.2.0',
    'sentence-transformers==5.7.0',
]
if not requirements_file.exists():
    with open(requirements_file, 'w') as f:
        f.write('\n'.join(requirements))
        print(f'{requirements_file} created')
else:
    print(f'{requirements_file} exists')

print('Установка зависимостей...')
!pip install -q -r requirements.txt
print('Зависимости успешно установлены!')

requirements.txt exists
Установка зависимостей...

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Зависимости успешно установлены!


In [6]:
import random

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer, pipeline

/Users/ngsmirnov/nikki/projects/practicum/sprint28-neural-networks-pytorch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
seed = 627
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [8]:
print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if hasattr(torch.backends, 'mps'):
    print(f'MPS available: {torch.backends.mps.is_available()}')

torch 2.9.1
CUDA available: False
MPS available: True


## Токенизация текста

In [9]:
# model_name = 'cointegrated/rubert-tiny2'
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# text = 'Apple stock rose 3% after CEO Tim Cook spoke in Cupertino.'
# encoded = tokenizer(text, return_tensors='pt')

# print('input_ids shape:', encoded['input_ids'].shape)
# print('tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))

## Эмбеддинги BERT (pooler output)

In [10]:
# model = AutoModel.from_pretrained(model_name)
# model.eval()

# with torch.no_grad():
#     outputs = model(**encoded)

# last_hidden = outputs.last_hidden_state
# pooled = outputs.pooler_output
# print('last_hidden_state:', last_hidden.shape)
# print('pooler_output:', pooled.shape)

## NER на веб-подобном тексте (готовый пайплайн)

In [11]:
# ner = pipeline(
#     'ner',
#     model='dslim/bert-base-NER',
#     aggregation_strategy='simple',
# )

# web_snippet = (
#     'Microsoft announced a partnership with OpenAI in Seattle. '
#     'Users on reddit.com debated the deal until late Tuesday.'
# )

# for entity in ner(web_snippet):
#     print(
#         f"{entity['word']:15} "
#         f"{entity['entity_group']:8} "
#         f"score={entity['score']:.3f}"
#     )

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("DeepPavlov/rubert-base-cased")
tokenizer = model.tokenizer

sentence_a = "Он открыл замок ключом"
sentence_b = "Дверь была заперта весь день"

encoded = tokenizer(
    sentence_a, sentence_b,
    return_tensors="pt", padding="max_length", max_length=16,
)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
token_type_ids = encoded["token_type_ids"][0]
attention_mask = encoded["attention_mask"][0]

print(tokens)
print(token_type_ids.tolist())
print(attention_mask.tolist())

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['[CLS]', 'Он', 'открыл', 'замок', 'ключом', '[SEP]', 'Двер', '##ь', 'была', 'заперт', '##а', 'весь', 'день', '[SEP]', '[PAD]', '[PAD]']
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]


In [33]:
sentence_a = "Врач осмотрел пациента утром"
sentence_b = "Пациент жаловался на головную боль"

# Токенизация пары предложений
encoded = tokenizer(
    sentence_a, sentence_b,
    return_tensors="pt", padding="max_length", max_length=16,
)

# Преобразование ID токенов в их текстовое представление
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

# Получение идентификаторов сегментов (0 — первое предложение, 1 — второе)
token_type_ids = encoded["token_type_ids"][0].tolist()

# Получение маски внимания (1 — реальный токен, 0 — [PAD])
attention_mask = encoded["attention_mask"][0].tolist()

# Вывод токенов, token_type_ids и attention_mask
print(tokens)
print(token_type_ids)
print(attention_mask)

# tokens, token_type_ids, attention_mask уже определены выше
print(encoded)
# Найдите индекс специального токена [CLS]
cls_index = tokens.index('[CLS]') #min(i for i,t in enumerate(tokens) if t == '[CLS]')
# Найдите индексы всех специальных токенов [SEP]
sep_indices = [i for i,t in enumerate(tokens) if t in ('[SEP]')]

# Вычислите длину первого предложения (без [CLS], [SEP] и [PAD])
sentence_a_length = sum([1 for t in tokens[:sep_indices[0]] if t not in ('[CLS]', '[SEP]', '[PAD]')])

# Вычислите длину второго предложения (без [SEP] и [PAD])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
sentence_b_length = sum([1 for t in tokens[sep_indices[0]:] if t not in ('[CLS]', '[SEP]', '[PAD]')])

# Вывод результатов
print("Позиция [CLS]:", cls_index)
print("Позиции [SEP]:", sep_indices)
print("Длина первого предложения:", sentence_a_length)
print("Длина второго предложения:", sentence_b_length)

['[CLS]', 'Врач', 'осмотрел', 'пациента', 'утром', '[SEP]', 'Паци', '##ент', 'жаловался', 'на', 'голов', '##ную', 'боль', '[SEP]', '[PAD]', '[PAD]']
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
{'input_ids': tensor([[   101,  34818, 100752,  39325,  16833,    102,  83904,   1762,  68544,
           1469,   9882,   3065,   9725,    102,      0,      0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])}
['[CLS]', 'Врач', 'осмотрел', 'пациента', 'утром', '[SEP]', 'Паци', '##ент', 'жаловался', 'на', 'голов', '##ную', 'боль', '[SEP]', '[PAD]', '[PAD]']
Позиция [CLS]: 0
Позиции [SEP]: [5, 13]
Длина первого предложения: 4
Длина второго предложения: 7


In [42]:
pairs = {
    "А (ключ)": ("В кармане лежал ключ от квартиры", "ключ", "В горах бил чистый ключ", "ключ"),
    "Б (лук)": ("На грядке рос зелёный лук", "лук", "Охотник держал в руках тугой лук", "лук"),
    "В (коса)": ("У девушки была длинная коса", "коса", "В море далеко виднелась песчаная коса", "коса"),
}

auto_model = model[0].auto_model
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
auto_model = auto_model.to(device)

# Переводим модель в режим инференса
auto_model.eval()

# pairs, get_token_vector(sentence, target_token), cosine(v1, v2) уже определены в скрытой части кода
# Функция вычисления косинусного сходства между двумя векторами
def cosine(v1, v2):
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

# Получение контекстного эмбеддинга указанного токена
def get_contextual_vector(sentence, target_token):
    # Токенизируем предложение
    inputs = tokenizer(sentence, return_tensors="pt")

    # 2. Переносим все входные тензоры на то же устройство
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Преобразуем идентификаторы обратно в токены
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Находим позицию интересующего токена
    idx = tokens.index(target_token)

    # Пропускаем предложение через BERT
    with torch.no_grad():
        outputs = auto_model(**inputs)

    # Возвращаем контекстный эмбеддинг нужного токена
    return outputs.last_hidden_state[0, idx]

results = {}
for label, (s1, w1, s2, w2) in pairs.items():
    v1 = get_contextual_vector(s1, w1)  # контекстный вектор слова w1 в предложении s1
    v2 = get_contextual_vector(s2, w2)   # контекстный вектор слова w2 в предложении s2
    results[label] = round(cosine(v1, v2), 2)

for label, score in results.items():
    print(f"{label}: {score}")

А (ключ): 0.7
Б (лук): 0.51
В (коса): 0.42


In [43]:
model.eval()

texts = [
    "Великолепный фильм! Режиссёр мастерски выстроил напряжение.",
    "Отличная картина, режиссура на высшем уровне.",
    "Ужасная скучная поделка, потерял два часа жизни.",
    "Шедевр мирового кинематографа, смотрел не отрываясь.",
]

encoded = tokenizer(
    texts,
    padding=True,        # Выравниваем все тексты по длине
    truncation=True,     # Обрезаем, если длина превышает max_length
    max_length=128,
    return_tensors="pt"  # Возвращаем torch.Tensor
)

print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

torch.Size([4, 14])
torch.Size([4, 14])


Здесь 4 — это количество текстов в батче, а 14 — длина самого длинного текста после токенизации.